In [ ]:
import requests
import json
import time
import os
import pandas as pd

API_URL = "https://en.wikipedia.org/w/api.php"

file_names = ['wikipedia_arbitration_links_pg1.txt', 'wikipedia_arbitration_links_pg2.txt', 'wikipedia_arbitration_links_pg3.txt']
content = {}

for i, file_name in enumerate(file_names):
    if os.path.exists(file_name):
        with open(file_name, 'r', encoding='utf-8') as file:
            content[file_name] = file.read()
    else:
        print(f"The file {file_name} does not exist.")

# Create a list to store the extracted values
data = []

# Iterate over the content dictionary
for file_name, content_str in content.items():
    # Use regular expressions to extract the href and title values
    matches = re.findall(r'<a href="([^"]+)" title="([^"]+)"', content_str)
    
    # Iterate over the matches
    for match in matches:
        href = match[0]
        title = match[1]
        # Append the values to the data list
        data.append({'file': file_name, 'href': href, 'title': title})

# Create a pandas DataFrame from the data list
df = pd.DataFrame(data)

df['title_clean_api'] = df['title'].apply(lambda x: x.replace(' ', '_'))

# Load existing data if available
try:
    with open("arbitration_pages.json", "r", encoding="utf-8") as f:
        all_pages_data = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    all_pages_data = []

# Replace with your actual list of page titles
lst = df['title_clean_api'].tolist()

def chunk_list(lst, n=50):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

headers = {
    "User-Agent": "Katkell (katkell10@gmail.com)"
}

for chunk in chunk_list(lst, 50):
    params = {
        "action": "query",
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "titles": "|".join(chunk),
        "format": "json",
        "formatversion": "2",
        "redirects": "1"
    }

    resp = requests.get(API_URL, headers=headers, params=params)
    if resp.status_code != 200:
        print("HTTP error:", resp.status_code)
        time.sleep(1)
        continue

    data = resp.json()
    pages = data.get("query", {}).get("pages", [])

    for page in pages:
        title = page.get("title", "")
        revisions = page.get("revisions", [])
        wikitext = ""
        if revisions and "slots" in revisions[0]:
            wikitext = revisions[0]["slots"]["main"].get("content", "")

        # Only add if not already saved
        if not any(d.get("title") == title for d in all_pages_data):
            all_pages_data.append({
                "title": title,
                "wikitext": wikitext
            })
            print(f"Saved: {title}")

    # Write progress back to disk
    with open("arbitration_pages.json", "w", encoding="utf-8") as f:
        json.dump(all_pages_data, f, indent=2)

    # Be polite with API
    time.sleep(0.5)

print(f"Done! Total saved: {len(all_pages_data)}")
